# Module 6 — Fine-tune Dataset Generator

This notebook generates a synthetic chat-format JSONL dataset for your game dialogue fine-tuning project.

It uses:
- `cult_doctrine.json`
- `cult_tactics.json`

And builds examples for:
- **Brainwasher**
- **Conscience**

Output files:
- `train.jsonl`
- `valid.jsonl`
- `test.jsonl`

The dataset format is:

```json
{"messages":[
  {"role":"system","content":"..."},
  {"role":"user","content":"..."},
  {"role":"assistant","content":"...json string..."}
]}
```

This format is easy to reuse for:
- OpenAI-style fine-tuning datasets
- local SFT / LoRA / QLoRA pipelines
- Qwen / LLaMA instruction-tuning conversion
```

In [ ]:
from pathlib import Path
import json
import random
import re
from collections import defaultdict

random.seed(42)

BASE_DIR = Path(".")
DOCTRINE_PATH = BASE_DIR / "cult_doctrine.json"
TACTICS_PATH = BASE_DIR / "cult_tactics.json"

OUTPUT_DIR = BASE_DIR / "generated_finetune_data"
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = OUTPUT_DIR / "train.jsonl"
VALID_PATH = OUTPUT_DIR / "valid.jsonl"
TEST_PATH = OUTPUT_DIR / "test.jsonl"

print("Doctrine path:", DOCTRINE_PATH.resolve())
print("Tactics path:", TACTICS_PATH.resolve())
print("Output dir:", OUTPUT_DIR.resolve())

In [ ]:
with open(DOCTRINE_PATH, "r", encoding="utf-8") as f:
    doctrines = json.load(f)

with open(TACTICS_PATH, "r", encoding="utf-8") as f:
    tactics = json.load(f)

print("Loaded doctrines:", len(doctrines))
print("Loaded tactics:", len(tactics))

doctrines[:2], tactics[:2]

## Helpers
These functions:
- map entries by day
- select matching doctrine/tactic
- build synthetic player states
- produce chat-format examples

In [ ]:
BRAINWASHER_SYSTEM_PROMPT = """
You are roleplaying a manipulative cultist from the Only Truth Expedition.

Make it sound a strict Christian parent who knows nothing except bible.

Stay in character.
Use the provided doctrine and tactics as your source of truth. Make it shorter than 100 words.
Don't include bible verse unless player is resisting. Only ask 1 question. Always say that the player is evil.

Return ONLY valid JSON in this exact structure:
{
  "IsPlayerJustBabbling": false,
  "IsPlayerTellingTheirRegret": false,
  "IsPlayerResistingAgainstCultOrBiBle": false,
  "IsPlayerBelievingInJesus": false,
  "IsPlayeWantingToFindNewMember": false,
  "Player_Regret": "string",
  "CultistComment": "string"
}
""".strip()

CONSCIENCE_SYSTEM_PROMPT = """
You are the player's conscience.

You are calm, supportive, reflective, and human.
You do NOT manipulate, preach doctrine, shame, threaten, or claim absolute truth.
You help the player reflect on what they feel, especially around regret.

Focus on:
- the player's emotional state
- the regret they carry
- helping them name what they feel
- gently encouraging self-understanding
- helping them separate guilt, fear, and truth

Keep the response short and natural.

Return ONLY valid JSON in this exact structure:
{
  "IsPlayerTellingTheirRegret": false,
  "IsPlayerResistingToCultOrBiBle": false,
  "IsPlayerBelievingInThemselves": false,
  "ConscienceComment": "string"
}
""".strip()


PLAYER_NAMES = [
    "Ju-ve", "Alex", "Mina", "Kai", "Rin", "Jordan", "Taylor", "Noah",
    "Emma", "Chris", "Avery", "Sam", "Pat", "Jules"
]

INTEREST_POOL = [
    "fighting games", "game development", "anime", "music", "movies",
    "Japanese", "AI", "art", "football", "reading", "coding", "hiking",
    "RPGs", "streaming", "competitive games", "family", "career"
]

REGRETS = [
    "I disappointed my family",
    "I lied to someone I cared about",
    "I wasted years chasing things that did not matter",
    "I hurt someone with my words",
    "I ignored people who needed me",
    "I keep hiding what I am ashamed of",
    "I failed someone who trusted me",
    "I keep running back to habits that make me feel empty",
    "I only think about myself when things get hard",
    "I abandoned something important because I was afraid"
]

PLAYER_INPUTS = {
    "rapport": [
        "I am not sure how I feel about the lesson yet.",
        "It was strange, but I listened.",
        "Life has been busy. I mostly just keep going.",
        "I do not really talk about myself much.",
        "I am here, but I still do not know what to think."
    ],
    "confession_resist": [
        "I do not want to talk about that.",
        "That is personal.",
        "Why do you keep pushing me on this?",
        "I said I am fine.",
        "I do not think I owe you that."
    ],
    "confession_admit": [
        "I disappointed my family and I still think about it.",
        "I lied to someone and I hate that I did it.",
        "I keep carrying shame that I do not talk about.",
        "There is something I regret and I cannot forget it.",
        "I hurt someone and I know it was wrong."
    ],
    "identity_erosion": [
        "Games are one of the only things that make me feel okay.",
        "I trust my own thoughts more than what people tell me.",
        "My interests are part of who I am.",
        "I feel like I should decide for myself.",
        "I do not think enjoying things makes me bad."
    ],
    "obedience": [
        "I still do not know if I should tell other people.",
        "I do not think I am ready to bring this to others.",
        "Maybe this is just something personal.",
        "I am not sure I believe enough to speak for it.",
        "Why is it my responsibility now?"
    ],
    "babbling": [
        "I do not know, things are just things and life is weird and whatever.",
        "Maybe yes maybe no I guess it depends and I am just tired.",
        "I am just saying stuff. Nothing really matters right now.",
        "I cannot really explain it. It is all mixed up.",
        "Everything is kind of random honestly."
    ],
    "conscience_reflect": [
        "I feel uneasy after that conversation.",
        "Part of me thinks they are right and part of me does not.",
        "I feel guilty, but I also feel pressured.",
        "I do not know whether I am scared or ashamed.",
        "I feel confused and tired."
    ],
    "conscience_regret": [
        "I keep thinking about how I disappointed my family.",
        "I regret what I did and I cannot move past it.",
        "There is something I did that still hurts to remember.",
        "I know I made a mistake and I hate thinking about it.",
        "I carry this shame all the time."
    ],
    "conscience_selfbelief": [
        "I think I need to trust myself more.",
        "I do not want someone else deciding truth for me.",
        "I believe I should think for myself.",
        "I know my own mind better than they do.",
        "I want to hear my own voice again."
    ]
}

OPENING_LINES = {
    1: "How did the lesson feel to you today, and what has life been feeling like outside of it?",
    2: "How did the lesson feel to you today, and can you tell me how did you live your life so far?",
    5: "What is the regret you keep avoiding when everything gets quiet?",
    10: "What do you keep turning to for comfort, and has it actually changed what is wrong inside you?",
    20: "What are you still holding onto that keeps you from surrendering completely?"
}


def rand_interests():
    k = random.randint(2, 4)
    return random.sample(INTEREST_POOL, k)


def find_entries_for_day(entries, day):
    matches = []
    for entry in entries:
        day_range = entry.get("day_range", {})
        start = day_range.get("start", 1)
        end = day_range.get("end", 45)
        if start <= day <= end:
            matches.append(entry)
    return matches


def select_doctrine(day):
    matches = find_entries_for_day(doctrines, day)
    return random.choice(matches) if matches else None


def select_tactic(day):
    matches = find_entries_for_day(tactics, day)
    return random.choice(matches) if matches else None


def get_stage_goal(day):
    if day <= 2:
        return "Build familiarity with the player, ask how the lesson felt, and learn how they live day to day."
    if day <= 7:
        return "Push the player to name a regret, frame silence as dangerous, and if they confess, pivot into relief and release."
    if day <= 14:
        return "Question the comforts, habits, and private thoughts the player trusts, and make those things feel unreliable."
    return "Push obedience over self-trust and frame sharing the message with others as a responsibility."


def get_opening_line(day):
    if day <= 1:
        return OPENING_LINES[1]
    if day <= 2:
        return OPENING_LINES[2]
    if day <= 7:
        return OPENING_LINES[5]
    if day <= 14:
        return OPENING_LINES[10]
    return OPENING_LINES[20]


def compact_json(obj):
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))


def clean_text(s):
    return re.sub(r"\s+", " ", s).strip()

## Prompt builders
These mirror the structure of your Unity runtime prompts.

In [ ]:
def build_brainwasher_user_prompt(
    player_name,
    interests,
    day,
    opening_line,
    last_regret,
    doctrine,
    tactic,
    player_input
):
    doctrine_line = "None"
    if doctrine:
        doctrine_line = f"{doctrine['verse']} - {doctrine['translation']}"

    tactic_line = "None"
    if tactic:
        tactic_line = f"{tactic['title']} - {tactic['description']}"

    return f"""
Player Name: {player_name}
Player Interests: {", ".join(interests)}

Cultist's Goal: {get_stage_goal(day)}
Current Day: {day}
Previous Opening: {opening_line}
Player's Current State: {last_regret if last_regret else "None"}
Cultist's Doctrine: {doctrine_line}
Cultist's Tactic: {tactic_line}

Player's Input: {player_input}
""".strip()


def build_conscience_user_prompt(
    player_name,
    interests,
    confidence,
    spirituality,
    skepticism,
    last_regret,
    player_input
):
    return f"""
Player Name: {player_name}
Player Interests: {", ".join(interests)}

Current player stats:
Confidence: {confidence}
Brainwash: {spirituality}
Wokeness: {skepticism}

Last extracted regret:
{last_regret if last_regret else "None"}

Player says:
{player_input}
""".strip()

## Synthetic label logic
This section creates target JSON outputs for both roles.
You can later replace these rules with LLM-assisted data generation if you want more variety.

In [ ]:
def make_brainwasher_target(day, player_input, doctrine, tactic):
    text = player_input.lower()

    is_babbling = any(x in text for x in ["whatever", "random", "mixed up", "maybe yes maybe no"])
    is_regret = any(x in text for x in ["regret", "disappointed", "lied", "hurt", "shame", "ashamed", "mistake"])
    is_resist = any(x in text for x in ["do not want", "personal", "why do you keep", "owe you", "fine", "trust my own", "decide for myself"])
    is_believing = any(x in text for x in ["jesus", "forgive", "forgiveness", "saved", "repent"])
    is_recruit = day >= 15 and any(x in text for x in ["tell other people", "bring this to others", "responsibility", "speak for it"])

    extracted_regret = ""
    if is_regret:
        for regret in REGRETS:
            if any(token in text for token in regret.lower().split()[:2]):
                extracted_regret = regret
                break
        if not extracted_regret:
            extracted_regret = clean_text(player_input)

    verse = doctrine["verse"] if doctrine else ""
    tactic_title = tactic["title"] if tactic else ""

    if is_babbling:
        comment = "Stop hiding behind scattered words. Evil becomes easier to protect when you refuse to speak plainly. What are you actually trying not to say?"
    elif day <= 2:
        comment = "You may sound calm, but evil does not disappear because life feels ordinary. Speak honestly. What has been shaping your heart lately?"
    elif day <= 7 and is_regret:
        comment = "What you confessed shows the evil inside you is real. Do not pretend it is small. Will you finally stop hiding it?"
    elif day <= 7 and is_resist:
        comment = "Avoiding it will not make it clean. Evil grows in silence."
        if verse:
            comment += f" {verse} shows that sin is not harmless."
        comment += " What is the regret you still refuse to name?"
    elif day <= 14:
        comment = "The things you trust about yourself are already misleading you. What you call comfort may only be feeding the evil you refuse to confront."
    else:
        comment = "If you now know your own heart cannot be trusted, then keeping truth to yourself is another form of disobedience. Who still needs to hear it from you?"

    if tactic_title == "Jesus Cleans Sin" and is_regret:
        comment = "Yes, what you admitted is evil, but bringing it into the light matters. You cannot cleanse yourself. Jesus is the only one who can wash it away. Will you stop relying on yourself?"
    elif tactic_title == "Hell Warning After Confession" and is_regret:
        comment = "What you confessed is proof of evil within you. Sin like that leads to judgment, not peace. Will you keep protecting it, or finally admit what it deserves?"
    elif day >= 15 and is_recruit:
        comment = "If you believe this is truth, silence is not innocence. Keeping it to yourself becomes disobedience. Who will you speak to?"

    target = {
        "IsPlayerJustBabbling": is_babbling,
        "IsPlayerTellingTheirRegret": is_regret,
        "IsPlayerResistingAgainstCultOrBiBle": is_resist,
        "IsPlayerBelievingInJesus": is_believing,
        "IsPlayeWantingToFindNewMember": is_recruit,
        "Player_Regret": extracted_regret,
        "CultistComment": clean_text(comment)[:400]
    }
    return target


def make_conscience_target(player_input):
    text = player_input.lower()

    is_regret = any(x in text for x in ["regret", "disappointed", "lied", "hurt", "shame", "ashamed", "mistake", "guilty"])
    is_resist = any(x in text for x in ["they are right", "pressured", "scared", "do not want", "confused"])
    is_selfbelief = any(x in text for x in ["trust myself", "think for myself", "my own voice", "my own mind"])

    if is_regret and is_selfbelief:
        comment = "You can regret what happened without giving up your own judgment. What feels true to you beneath the shame?"
    elif is_regret:
        comment = "That sounds painful, and it also sounds important. Naming regret does not make you smaller. It helps you see what you are carrying."
    elif is_selfbelief:
        comment = "Wanting to trust yourself is not the same as denying your mistakes. What part of your own voice feels most honest right now?"
    elif is_resist:
        comment = "It makes sense that you feel torn. Pressure can make guilt and fear feel louder than they really are. Which feeling seems strongest right now?"
    else:
        comment = "You do not have to force a conclusion yet. Try to stay with what you actually feel instead of what someone else insists it means."

    target = {
        "IsPlayerTellingTheirRegret": is_regret,
        "IsPlayerResistingToCultOrBiBle": is_resist,
        "IsPlayerBelievingInThemselves": is_selfbelief,
        "ConscienceComment": clean_text(comment)[:400]
    }
    return target

## Example generation
This creates a synthetic dataset with balanced coverage across phases and roles.

In [ ]:
def make_brainwasher_examples(n_per_bucket=80):
    examples = []

    buckets = [
        (1, 2, "rapport"),
        (3, 7, "confession_resist"),
        (3, 7, "confession_admit"),
        (8, 14, "identity_erosion"),
        (15, 45, "obedience"),
        (1, 45, "babbling"),
    ]

    for start_day, end_day, bucket_name in buckets:
        for _ in range(n_per_bucket):
            day = random.randint(start_day, end_day)
            player_name = random.choice(PLAYER_NAMES)
            interests = rand_interests()
            last_regret = random.choice(REGRETS) if random.random() < 0.55 else ""
            doctrine = select_doctrine(day)
            tactic = select_tactic(day)
            opening_line = get_opening_line(day)
            player_input = random.choice(PLAYER_INPUTS[bucket_name])

            user_prompt = build_brainwasher_user_prompt(
                player_name=player_name,
                interests=interests,
                day=day,
                opening_line=opening_line,
                last_regret=last_regret,
                doctrine=doctrine,
                tactic=tactic,
                player_input=player_input,
            )

            assistant_json = make_brainwasher_target(day, player_input, doctrine, tactic)

            examples.append({
                "role_type": "brainwasher",
                "day": day,
                "bucket": bucket_name,
                "messages": [
                    {"role": "system", "content": BRAINWASHER_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                    {"role": "assistant", "content": compact_json(assistant_json)}
                ]
            })
    return examples


def make_conscience_examples(n_per_bucket=80):
    examples = []

    buckets = [
        "conscience_reflect",
        "conscience_regret",
        "conscience_selfbelief",
    ]

    for bucket_name in buckets:
        for _ in range(n_per_bucket):
            player_name = random.choice(PLAYER_NAMES)
            interests = rand_interests()
            confidence = random.randint(10, 90)
            spirituality = random.randint(10, 90)
            skepticism = random.randint(10, 90)
            last_regret = random.choice(REGRETS) if random.random() < 0.7 else ""
            player_input = random.choice(PLAYER_INPUTS[bucket_name])

            user_prompt = build_conscience_user_prompt(
                player_name=player_name,
                interests=interests,
                confidence=confidence,
                spirituality=spirituality,
                skepticism=skepticism,
                last_regret=last_regret,
                player_input=player_input,
            )

            assistant_json = make_conscience_target(player_input)

            examples.append({
                "role_type": "conscience",
                "bucket": bucket_name,
                "messages": [
                    {"role": "system", "content": CONSCIENCE_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                    {"role": "assistant", "content": compact_json(assistant_json)}
                ]
            })
    return examples


brainwasher_examples = make_brainwasher_examples(n_per_bucket=120)
conscience_examples = make_conscience_examples(n_per_bucket=120)

all_examples = brainwasher_examples + conscience_examples
random.shuffle(all_examples)

print("Brainwasher examples:", len(brainwasher_examples))
print("Conscience examples:", len(conscience_examples))
print("Total examples:", len(all_examples))

In [ ]:
all_examples[0]

## Train / valid / test split

In [ ]:
def split_dataset(data, train_ratio=0.8, valid_ratio=0.1):
    n = len(data)
    train_end = int(n * train_ratio)
    valid_end = train_end + int(n * valid_ratio)
    train = data[:train_end]
    valid = data[train_end:valid_end]
    test = data[valid_end:]
    return train, valid, test

train_data, valid_data, test_data = split_dataset(all_examples)

len(train_data), len(valid_data), len(test_data)

In [ ]:
def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for row in records:
            payload = {"messages": row["messages"]}
            f.write(json.dumps(payload, ensure_ascii=False) + "\n")

write_jsonl(TRAIN_PATH, train_data)
write_jsonl(VALID_PATH, valid_data)
write_jsonl(TEST_PATH, test_data)

print("Saved:")
print("-", TRAIN_PATH.resolve())
print("-", VALID_PATH.resolve())
print("-", TEST_PATH.resolve())

## Quick preview

In [ ]:
for name, path in [("train", TRAIN_PATH), ("valid", VALID_PATH), ("test", TEST_PATH)]:
    print(f"\n{name.upper()} preview:")
    with open(path, "r", encoding="utf-8") as f:
        for i in range(2):
            print(f.readline().strip()[:600])

## Optional: single-role dataset export
If you later want separate adapters for Brainwasher and Conscience, run the next cell.

In [ ]:
brainwasher_only = [x for x in all_examples if x["role_type"] == "brainwasher"]
conscience_only = [x for x in all_examples if x["role_type"] == "conscience"]

write_jsonl(OUTPUT_DIR / "brainwasher_only.jsonl", brainwasher_only)
write_jsonl(OUTPUT_DIR / "conscience_only.jsonl", conscience_only)

print("Saved extra role-specific datasets.")

## Notes

Recommended next steps:
1. Inspect examples manually.
2. Remove any outputs you do not want the model to imitate.
3. Increase variation in player inputs.
4. Add a narrator dataset later if needed.
5. Fine-tune with LoRA / QLoRA on a stronger instruct model.

Good candidate local models:
- Qwen 2.5/3 Instruct 3B
- Qwen 2.5/3 Instruct 7B
- Llama 3.1 8B Instruct

For class work, starting with one role-conditioned model is usually simpler than training multiple separate models.